### explore metadata of the generated synthetic datasets

In [ ]:
import pandas as pd

In [ ]:
import os
from pathlib import Path
import numpy as np

In [ ]:
DATA_ROOT = Path(os.environ["EPHEMERAL"]) / "pyspoc_synthetic_data_28oct2026"

syn_path_base = 'hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/metadata_files/'
syn_path = Path(syn_path_base).resolve() / 'raw_files/manifest_28oct2026.csv'
syn_data_df = pd.read_csv(syn_path)
# syn_data_df

In [ ]:
syn_data_df.shape

In [ ]:
syn_data_df['safe_data_id'] = syn_data_df['dataset_id'].map(lambda v : str(v).replace('/','__'))
syn_data_df['path'] = syn_data_df['dataset_id'].map(lambda v : DATA_ROOT / f'{v}.npy')

In [ ]:
syn_data_df[syn_data_df['n_rows'].isna()]

In [ ]:
# meta.shape[0] == syn_data_df.shape[0]

In [ ]:
unique_n_p_vals = np.unique(syn_data_df[['n_rows','n_cols']].dropna().values,axis=0)
print(unique_n_p_vals)

In [ ]:
# syn_data_df['setting']=meta['setting']

In [ ]:
for n,p in unique_n_p_vals:
    print(n,p)
    np_df = syn_data_df[(syn_data_df['n_rows']==n) & (syn_data_df['n_cols']==p)]
    nf=int(n); pf=int(p)
    print('saving..' + f'syn_data_N{nf}_P{pf}.csv')
    np_df.to_csv(Path(syn_path_base) / f'syn_data_N{nf}_P{pf}_num_seed_10.csv', index=None)

In [ ]:
# np_df

### getting correct dataset grouping settings from the configs.json


In [ ]:
import json

registry = json.load(open(Path(syn_path_base).resolve() / 'raw_files/configs_28oct2026.json'))

params_df = pd.concat(
    {gen: pd.DataFrame(lst, index=[f"params{i}" for i in range(len(lst))]) for gen, lst in registry.items()},
    names=["generator", "params_id"])

In [ ]:
# registry.items()[]

In [ ]:
syn_data_df.groupby(['generator'])['params_id'].nunique().sum()

In [ ]:
meta = syn_data_df.join(params_df, on=["generator", "params_id"])

In [ ]:
# meta

In [ ]:
rand_cols = [c for c in params_df.columns if "random_state" in c or "seed" in c]
rand_cols += ['cluster_std_min_max']

In [ ]:
setting_cols = [c for c in params_df.columns if c not in rand_cols]

In [ ]:
parts = [(c + "=" + meta[c].astype(str)).where(meta[c].notna(), "") for c in setting_cols]
key = parts[0].str.cat(parts[1:], sep="|").str.replace(r"\|+", "|", regex=True).str.strip("|")
meta["setting"] = meta["generator"] + "__" + key

In [ ]:
# meta.to_csv(index=None)

In [ ]:
# meta["setting"]

In [ ]:
# n=1000; p=100

for n,p in unique_n_p_vals:
    meta_sub = meta[(meta['n_rows']==n) & (meta['n_cols']==p)]

    sizes = meta_sub.groupby(["generator", "setting"]).size()
    mv = sizes.groupby("generator").describe()[["count", "min", "50%", "max"]].min()['min']
    
    print(mv)

In [ ]:
# meta["setting"] = meta["generator"] + "__" + 
# meta[setting_cols].fillna('').astype(str).agg("|".join, axis=1)
# sizes = meta.groupby(["generator", "setting"]).size()
# print(sizes.groupby("generator").describe()[["count", "min", "50%", "max"]])

In [ ]:
# hashable = meta[setting_cols].map(lambda v: str(v) if isinstance(v, (list, dict, np.ndarray)) else v)
# hashable["generator"] = meta["generator"]
# n = hashable.groupby("generator").size()
# uniq = hashable.groupby("generator")[setting_cols].nunique()
# # print(uniq.sum(axis=1))
# suspect = uniq.ge(0.5 * n.values[:, None])
# suspect = uniq.gt(10)   # nearly one value per dataset
# print(suspect.apply(lambda r: list(r.index[r]), axis=1).loc[lambda s: s.str.len() > 0])